# Multimodal Garbage Classifier — Google Colab T4 GPU
### CNN (EfficientNet-B0) + NLP (BERT) — Transfer Learning
| | |
|---|---|
| **Image branch** | EfficientNet-B0 pretrained on ImageNet |
| **Text branch** | BERT pretrained on Wikipedia + BooksCorpus |
| **Classes** | Black, Blue, Green, TTR |
| **Device** | T4 GPU (16GB VRAM) |

> **Before running:** Go to `Runtime → Change runtime type → T4 GPU`

## Cell 1 — Check GPU

In [ ]:
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name       : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU found! Go to Runtime -> Change runtime type -> T4 GPU')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted!')

## Cell 3 — Install Dependencies

In [ ]:
!pip install efficientnet_pytorch transformers tqdm -q
print('Done')

## Cell 4 — Imports

In [ ]:
import os
import re
import time
import copy
import torch
import torch.nn as nn
import numpy as np
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from transformers import BertTokenizer, BertModel
from efficientnet_pytorch import EfficientNet
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

print('All imports OK')

## Cell 5 — Configuration
> **Only edit this cell** — update paths to match your Google Drive structure

In [ ]:
# ── Update these to match your Drive folder structure ──
BASE_DIR  = '/content/drive/MyDrive/GarbageClassifier/balanced_dataset'
TRAIN_DIR = f'{BASE_DIR}/Train'
VAL_DIR   = f'{BASE_DIR}/Val'
TEST_DIR  = f'{BASE_DIR}/Test'
SAVE_DIR  = '/content/drive/MyDrive/GarbageClassifier/balancedModel'

# ── Classes ──
CLASSES      = ['Black', 'Blue', 'Green', 'TTR']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}

# ── Hyperparameters (tuned for T4 16GB) ──
BATCH_SIZE    = 32      # T4 has 16GB so we can use larger batch
NUM_EPOCHS    = 40
LR_FROZEN     = 1e-4   # LR when BERT is frozen (phase 1)
LR_UNFROZEN   = 1e-5   # LR when fine-tuning all layers (phase 2)
FREEZE_EPOCHS = 5       # freeze BERT for first N epochs
MAX_TEXT_LEN  = 32
IMAGE_SIZE    = 224
DROPOUT       = 0.4
PATIENCE      = 7
NUM_WORKERS   = 4       # more workers since Colab has more CPU cores
DEVICE        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Device : {DEVICE}')
print(f'Batch  : {BATCH_SIZE}')
print(f'Epochs : {NUM_EPOCHS}')
print(f'LR     : {LR_FROZEN} (frozen) -> {LR_UNFROZEN} (unfrozen)')
print()

# Sanity check paths
for label, d in [('Train', TRAIN_DIR), ('Val', VAL_DIR), ('Test', TEST_DIR)]:
    exists = os.path.exists(d)
    print(f'  {"OK" if exists else "MISSING"} {label}: {d}')

## Cell 6 — Dataset Class
Text description is extracted from the **original image filename** — this feeds the NLP (BERT) branch.

In [ ]:
class GarbageDataset(Dataset):
    def __init__(self, root_dir, tokenizer, transform=None):
        self.samples   = []
        self.tokenizer = tokenizer
        self.transform = transform

        for class_name in CLASSES:
            class_dir = os.path.join(root_dir, class_name)
            if not os.path.exists(class_dir):
                print(f'  Warning: {class_dir} not found')
                continue
            label = CLASS_TO_IDX[class_name]
            files = [f for f in os.listdir(class_dir)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            for fname in files:
                img_path = os.path.join(class_dir, fname)
                # NLP input: extract description from original filename
                # e.g. 'plastic_bottle_123.jpg' -> 'plastic bottle 123'
                text = re.sub(r'[_.\-]', ' ', os.path.splitext(fname)[0])
                text = ' '.join(text.split()).lower()
                self.samples.append((img_path, text, label))
            print(f'  {class_name:<8}: {len(files)} images')

        print(f'  Total   : {len(self.samples)} samples')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, text, label = self.samples[idx]

        # CNN input — image tensor
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        # NLP input — tokenized filename text
        encoding = self.tokenizer(
            text,
            max_length=MAX_TEXT_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return (
            image,
            encoding['input_ids'].squeeze(0),
            encoding['attention_mask'].squeeze(0),
            torch.tensor(label, dtype=torch.long)
        )

print('Dataset class defined')
print()
print('How NLP text extraction works:')
print('  plastic_bottle_123.jpg  ->  "plastic bottle 123"')
print('  old-newspaper.jpg       ->  "old newspaper"')
print('  food_waste_456.png      ->  "food waste 456"')

## Cell 7 — Model Architecture
```
Image  -->  EfficientNet-B0 (ImageNet pretrained)  -->  1280-dim
                                                                   -->  Fusion  -->  4 bins
Text   -->  BERT-base (Wikipedia pretrained)       -->   768-dim
```

In [ ]:
class MultimodalGarbageClassifier(nn.Module):
    def __init__(self, num_classes=4, dropout=DROPOUT):
        super().__init__()

        # CNN Branch — EfficientNet-B0 pretrained on ImageNet
        self.image_encoder = EfficientNet.from_pretrained('efficientnet-b0')
        img_feat_dim = self.image_encoder._fc.in_features  # 1280
        self.image_encoder._fc = nn.Identity()             # remove original head

        # NLP Branch — BERT pretrained on Wikipedia + BooksCorpus
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        text_feat_dim = self.bert.config.hidden_size        # 768

        # Fusion: 1280 + 768 = 2048 -> 4 classes
        self.fusion = nn.Sequential(
            nn.Linear(img_feat_dim + text_feat_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(128, num_classes)
        )

    def forward(self, image, input_ids, attention_mask):
        # CNN: visual features
        img_feat  = self.image_encoder(image)                        # (B, 1280)
        # NLP: semantic features from filename text
        bert_out  = self.bert(input_ids=input_ids,
                              attention_mask=attention_mask)
        text_feat = bert_out.last_hidden_state[:, 0, :]              # (B, 768) CLS token
        # Fusion: combine both modalities
        fused  = torch.cat([img_feat, text_feat], dim=1)             # (B, 2048)
        return self.fusion(fused)                                    # (B, 4)

print('Model defined')
print('  CNN  : EfficientNet-B0 -> 1280 features')
print('  NLP  : BERT-base       ->  768 features')
print('  Fuse : 2048 -> 512 -> 256 -> 128 -> 4 classes')

## Cell 8 — Load Data

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(20),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print('Loading BERT tokenizer...')
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

print('\nTrain:')
train_dataset = GarbageDataset(TRAIN_DIR, tokenizer, train_transform)
print('\nVal:')
val_dataset   = GarbageDataset(VAL_DIR,   tokenizer, val_transform)
print('\nTest:')
test_dataset  = GarbageDataset(TEST_DIR,  tokenizer, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

print(f'\nTrain batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')
print(f'\nEstimated time per epoch on T4: ~8-15 min')
print(f'Estimated total for 40 epochs : ~5-10 hours')

## Cell 9 — Build Model & Load Pretrained Weights

In [ ]:
torch.cuda.empty_cache()

print('Building model with pretrained weights...')
model = MultimodalGarbageClassifier(num_classes=len(CLASSES)).to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FROZEN, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=2, factor=0.5
)
scaler = GradScaler()  # mixed precision

# Try loading existing checkpoint from Drive
best_model_path = os.path.join(SAVE_DIR, 'best_model.pth')
if os.path.exists(best_model_path):
    try:
        model.load_state_dict(torch.load(best_model_path, map_location=DEVICE, weights_only=True))
        print(f'Loaded existing checkpoint -> continuing fine-tuning')
    except RuntimeError:
        print(f'Architecture mismatch -> starting fresh with pretrained weights')
else:
    print(f'No checkpoint found -> starting fresh with pretrained weights')

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal params     : {total_params:,}')
print(f'Trainable params : {trainable_params:,}')
print(f'VRAM used        : {torch.cuda.memory_allocated()/1e9:.2f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## Cell 10 — Training Loop
**Phase 1** (epochs 1-5): BERT frozen → fast, only trains EfficientNet + Fusion

**Phase 2** (epoch 6+): Everything unfrozen → fine-tune all layers together

In [ ]:
best_model_wts   = copy.deepcopy(model.state_dict())
best_val_acc     = 0.0
patience_counter = 0
stop_training    = False
epoch_times      = []
history          = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print(f'Training on {DEVICE} | Batch: {BATCH_SIZE} | Mixed Precision: ON')
print(f'Phase 1: epochs 1-{FREEZE_EPOCHS}  -> BERT frozen   | LR={LR_FROZEN}')
print(f'Phase 2: epochs {FREEZE_EPOCHS+1}+ -> all unfrozen  | LR={LR_UNFROZEN}')
print()

for epoch in range(NUM_EPOCHS):
    if stop_training:
        break

    # Phase 1 — freeze BERT
    if epoch == 0:
        for param in model.bert.parameters():
            param.requires_grad = False
        for g in optimizer.param_groups:
            g['lr'] = LR_FROZEN
        print(f'Phase 1 started: BERT frozen | LR = {LR_FROZEN}')

    # Phase 2 — unfreeze BERT
    if epoch == FREEZE_EPOCHS:
        for param in model.bert.parameters():
            param.requires_grad = True
        for g in optimizer.param_groups:
            g['lr'] = LR_UNFROZEN
        print(f'\nPhase 2 started: BERT unfrozen | LR = {LR_UNFROZEN}')

    # ETA estimate
    if epoch_times:
        eta = time.strftime('%H:%M:%S', time.gmtime(np.mean(epoch_times) * (NUM_EPOCHS - epoch)))
    else:
        eta = 'calculating...'

    print(f'\nEpoch {epoch+1}/{NUM_EPOCHS}  |  ETA remaining: {eta}')
    print('-' * 65)
    epoch_start = time.time()

    for phase in ['train', 'val']:
        loader = train_loader if phase == 'train' else val_loader
        model.train() if phase == 'train' else model.eval()

        running_loss     = 0.0
        running_corrects = 0
        total_samples    = 0

        pbar = tqdm(
            loader,
            desc=f'  {phase.upper():5s}',
            ncols=95,
            leave=True,
            bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining} {rate_fmt}]'
        )

        for images, input_ids, attention_masks, labels in pbar:
            images          = images.to(DEVICE, non_blocking=True)
            input_ids       = input_ids.to(DEVICE, non_blocking=True)
            attention_masks = attention_masks.to(DEVICE, non_blocking=True)
            labels          = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast():
                outputs = model(images, input_ids, attention_masks)
                loss    = criterion(outputs, labels)

            preds = outputs.argmax(dim=1)

            if phase == 'train':
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()

            running_loss     += loss.item() * images.size(0)
            running_corrects += (preds == labels).sum().item()
            total_samples    += images.size(0)

            pbar.set_postfix({
                'loss': f'{running_loss/total_samples:.4f}',
                'acc' : f'{running_corrects/total_samples*100:.1f}%',
                'VRAM': f'{torch.cuda.memory_allocated()/1e9:.1f}GB'
            })

        epoch_loss = running_loss / total_samples
        epoch_acc  = running_corrects / total_samples
        history[f'{phase}_loss'].append(epoch_loss)
        history[f'{phase}_acc'].append(epoch_acc)
        print(f'  {phase.upper():5s} -> Loss: {epoch_loss:.4f} | Acc: {epoch_acc*100:.2f}%')

        if phase == 'val':
            scheduler.step(epoch_loss)
            if epoch_acc > best_val_acc:
                best_val_acc     = epoch_acc
                best_model_wts   = copy.deepcopy(model.state_dict())
                patience_counter = 0
                ckpt_path        = os.path.join(SAVE_DIR, 'best_model.pth')
                torch.save(model.state_dict(), ckpt_path)
                print(f'  Best model saved to Drive (Val Acc: {best_val_acc*100:.2f}%)')
            else:
                patience_counter += 1
                print(f'  No improvement ({patience_counter}/{PATIENCE})')
                if patience_counter >= PATIENCE:
                    print(f'  Early stopping at epoch {epoch+1}')
                    stop_training = True
                    break

    epoch_time = time.time() - epoch_start
    epoch_times.append(epoch_time)
    print(f'  Epoch time : {epoch_time/60:.1f} min')
    print(f'  VRAM used  : {torch.cuda.memory_allocated()/1e9:.2f} GB')
    torch.cuda.empty_cache()

model.load_state_dict(best_model_wts)
total_time = sum(epoch_times)
print(f'\nTraining complete!')
print(f'Best Val Accuracy : {best_val_acc*100:.2f}%')
print(f'Total time        : {total_time/3600:.1f} hours ({len(epoch_times)} epochs)')

## Cell 11 — Plot Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train', marker='o', linewidth=2)
axes[0].plot(history['val_loss'],   label='Val',   marker='o', linewidth=2)
axes[0].axvline(x=FREEZE_EPOCHS-1, color='red', linestyle='--',
                alpha=0.6, label=f'BERT unfrozen (ep {FREEZE_EPOCHS+1})')
axes[0].set_title('Loss per Epoch', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot([a*100 for a in history['train_acc']], label='Train', marker='o', linewidth=2)
axes[1].plot([a*100 for a in history['val_acc']],   label='Val',   marker='o', linewidth=2)
axes[1].axvline(x=FREEZE_EPOCHS-1, color='red', linestyle='--',
                alpha=0.6, label=f'BERT unfrozen (ep {FREEZE_EPOCHS+1})')
axes[1].set_title('Accuracy per Epoch', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('CNN (EfficientNet-B0) + NLP (BERT) — Garbage Classifier', fontsize=14, fontweight='bold')
plt.tight_layout()
path = os.path.join(SAVE_DIR, 'training_curves.png')
plt.savefig(path, dpi=150)
plt.show()
print(f'Saved to Drive -> {path}')

## Cell 12 — Save Final Model to Drive

In [ ]:
final_path = os.path.join(SAVE_DIR, 'balanced_final_model.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'class_to_idx'    : CLASS_TO_IDX,
    'classes'         : CLASSES,
    'best_val_acc'    : best_val_acc,
    'config': {
        'image_size'   : IMAGE_SIZE,
        'max_text_len' : MAX_TEXT_LEN,
        'dropout'      : DROPOUT,
        'architecture' : 'EfficientNet-B0 + BERT'
    }
}, final_path)
print(f'Final model saved to Drive -> {final_path}')
print(f'Best Val Accuracy          : {best_val_acc*100:.2f}%')

## Cell 13 — Evaluate on Test Set

In [ ]:
model.eval()
all_preds  = []
all_labels = []

print('Running on test set...')
with torch.no_grad():
    for images, input_ids, attention_masks, labels in tqdm(test_loader, desc='Testing', ncols=70):
        images          = images.to(DEVICE)
        input_ids       = input_ids.to(DEVICE)
        attention_masks = attention_masks.to(DEVICE)
        with autocast():
            outputs = model(images, input_ids, attention_masks)
        preds = outputs.argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.tolist())

correct  = sum(p == l for p, l in zip(all_preds, all_labels))
test_acc = correct / len(all_labels) * 100
print(f'\nTest Accuracy: {test_acc:.2f}%')
print('\nClassification Report:')
print(classification_report(all_labels, all_preds, target_names=CLASSES))

print('Per class accuracy:')
for i, cls in enumerate(CLASSES):
    cls_total   = all_labels.count(i)
    cls_correct = sum(p == l for p, l in zip(all_preds, all_labels) if l == i)
    print(f'  {cls:<8}: {cls_correct}/{cls_total}  ({cls_correct/cls_total*100:.1f}%)')

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES)
plt.title(f'Confusion Matrix (Test Acc: {test_acc:.1f}%)', fontsize=13)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
cm_path = os.path.join(SAVE_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'Saved to Drive -> {cm_path}')

## Cell 14 — Predict All Test Images + Show Misclassified

In [ ]:
results   = []
wrong     = []
color_map = {'Black': '#555555', 'Blue': '#3399ff', 'Green': '#33cc66', 'TTR': '#ff9900'}

model.eval()
for true_class in CLASSES:
    class_dir = os.path.join(TEST_DIR, true_class)
    files     = [f for f in os.listdir(class_dir)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    print(f'Predicting {true_class}: {len(files)} images...')

    for fname in tqdm(files, desc=f'  {true_class}', ncols=70):
        img_path = os.path.join(class_dir, fname)
        text     = re.sub(r'[_.\-]', ' ', os.path.splitext(fname)[0])
        text     = ' '.join(text.split()).lower()

        image      = Image.open(img_path).convert('RGB')
        img_tensor = val_transform(image).unsqueeze(0).to(DEVICE)
        encoding   = tokenizer(text, max_length=MAX_TEXT_LEN, padding='max_length',
                               truncation=True, return_tensors='pt')
        input_ids      = encoding['input_ids'].to(DEVICE)
        attention_mask = encoding['attention_mask'].to(DEVICE)

        with torch.no_grad():
            with autocast():
                logits = model(img_tensor, input_ids, attention_mask)
            probs = torch.softmax(logits.float(), dim=1).squeeze(0)

        pred_class = CLASSES[probs.argmax().item()]
        confidence = probs.max().item() * 100
        results.append({'true': true_class, 'pred': pred_class,
                        'correct': pred_class == true_class, 'text': text})
        if pred_class != true_class:
            wrong.append((fname, true_class, pred_class, confidence, image, text))

# Summary
total   = len(results)
correct = sum(r['correct'] for r in results)
print(f'\nOverall Accuracy: {correct}/{total}  ({correct/total*100:.2f}%)')
print()
for cls in CLASSES:
    cls_r = [r for r in results if r['true'] == cls]
    cls_c = sum(r['correct'] for r in cls_r)
    print(f'  {cls:<8}: {cls_c}/{len(cls_r)}  ({cls_c/len(cls_r)*100:.1f}%)')

# Show misclassified
if wrong:
    print(f'\nMisclassified: {len(wrong)} images')
    cols = 4
    rows = (len(wrong) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*4))
    axes = axes.flatten() if len(wrong) > 1 else [axes]
    for i, (fname, true_cls, pred_cls, conf, img, text) in enumerate(wrong):
        axes[i].imshow(img)
        axes[i].set_title(
            f'True: {true_cls}\nPred: {pred_cls} ({conf:.0f}%)\nText: "{text[:25]}"',
            fontsize=7, color=color_map[pred_cls]
        )
        axes[i].axis('off')
    for j in range(i+1, len(axes)):
        axes[j].axis('off')
    plt.suptitle(f'Misclassified Images ({len(wrong)} total)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    mis_path = os.path.join(SAVE_DIR, 'misclassified.png')
    plt.savefig(mis_path, dpi=150)
    plt.show()
    print(f'Saved to Drive -> {mis_path}')
else:
    print('No misclassified images!')

## Cell 15 — Single Image Prediction

In [ ]:
def predict_single(image_path, custom_text=None):
    if custom_text:
        text = custom_text.lower()
    else:
        fname = os.path.splitext(os.path.basename(image_path))[0]
        text  = re.sub(r'[_.\-]', ' ', fname)
        text  = ' '.join(text.split()).lower()

    image      = Image.open(image_path).convert('RGB')
    img_tensor = val_transform(image).unsqueeze(0).to(DEVICE)
    encoding   = tokenizer(text, max_length=MAX_TEXT_LEN, padding='max_length',
                           truncation=True, return_tensors='pt')
    input_ids      = encoding['input_ids'].to(DEVICE)
    attention_mask = encoding['attention_mask'].to(DEVICE)

    model.eval()
    with torch.no_grad():
        with autocast():
            logits = model(img_tensor, input_ids, attention_mask)
        probs = torch.softmax(logits.float(), dim=1).squeeze(0)

    pred_class = CLASSES[probs.argmax().item()]
    confidence = probs.max().item() * 100
    probs_dict = {cls: round(probs[i].item()*100, 2) for i, cls in enumerate(CLASSES)}

    color_map = {'Black': '#555555', 'Blue': '#3399ff', 'Green': '#33cc66', 'TTR': '#ff9900'}
    bin_info  = {
        'Black': 'General / Non-recyclable waste',
        'Blue' : 'Recyclable (paper, plastic, cans)',
        'Green': 'Organic / Compostable waste',
        'TTR'  : 'Take it to a Recycling depot'
    }

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].imshow(image)
    axes[0].set_title(
        f'Predicted: {pred_class} ({confidence:.1f}%)\n{bin_info[pred_class]}\nText: "{text}"',
        fontsize=11, color=color_map[pred_class], fontweight='bold'
    )
    axes[0].axis('off')

    colors = [color_map[c] for c in CLASSES]
    bars   = axes[1].bar(CLASSES, [probs_dict[c] for c in CLASSES], color=colors, edgecolor='black')
    axes[1].set_ylim(0, 100)
    axes[1].set_ylabel('Confidence (%)')
    axes[1].set_title('Class Probabilities')
    for bar, cls in zip(bars, CLASSES):
        axes[1].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + 1,
                     f'{probs_dict[cls]:.1f}%', ha='center', fontsize=10)
    plt.tight_layout()
    plt.show()

    print(f'Text used  : "{text}"')
    print(f'Prediction : {pred_class} -> {bin_info[pred_class]}')
    print(f'Confidence : {confidence:.1f}%')
    return pred_class, confidence


# Auto test with first image from each class
print('Testing one sample from each class:\n')
for cls in CLASSES:
    cls_dir = os.path.join(TEST_DIR, cls)
    sample  = os.listdir(cls_dir)[0]
    print(f'--- {cls}: {sample} ---')
    predict_single(os.path.join(cls_dir, sample))
    print()